# Offline Processing: Sentence-Level Audio Generation & Vector Storage

### Pipeline
1. Load sentences from `Sentence_with_phonemes.xlsx`
2. Use **g2p_en** for per-word phoneme extraction
3. Generate **full sentence** audio via Piper TTS
4. Transcribe with **Whisper** to get word-level timestamps
5. Chunk each word from the sentence audio
6. Embed each chunk with **Wav2Vec2** (768-d, mean pooling)
7. Store in **Pinecone** (namespace: `speech`, metric: cosine)


In [1]:
!pip install pandas openpyxl g2p-en torch librosa numpy openai-whisper transformers pinecone-client python-dotenv


In [2]:
import pandas as pd
from g2p_en import G2p

# Load sentences
df = pd.read_excel("Sentence_with_phonemes.xlsx")
sentences = df["Sentence"].dropna().unique().tolist()
print(f"Loaded {len(sentences)} sentences")

# Build word -> phoneme lookup using g2p
g2p = G2p()
word_phonemes = {}
for sentence in sentences:
    for word in sentence.split():
        clean = word.strip(".,!?;: ").lower()
        if clean and clean not in word_phonemes:
            phonemes = g2p(clean)
            phonemes = [p for p in phonemes if p != " "]
            word_phonemes[clean] = " ".join(phonemes)

print(f"Built {len(word_phonemes)} word-phoneme mappings")


Loaded 34 sentences
Built 191 word-phoneme mappings


In [3]:
import os
import subprocess

PIPER_PATH = r"D:\\INFOSYS\\.venv\\Scripts\\piper.exe"
VOICE_DIR = r"D:\\INFOSYS\\.venv\\share\\piper\\voices"
MODEL_PATH = os.path.join(VOICE_DIR, "en_US-lessac-medium.onnx")
CONFIG_PATH = os.path.join(VOICE_DIR, "en_US-lessac-medium.onnx.json")

def generate_audio_with_piper(text, output_file="raw.wav"):
    command = [
        "piper",
        "--model", MODEL_PATH,
        "--config", CONFIG_PATH,
        "--output_file", output_file
    ]
    process = subprocess.Popen(command, stdin=subprocess.PIPE, text=True)
    process.communicate(text + "\n")
    print(f"Audio generated: {output_file}")


In [4]:
# Generate full-sentence audio (NOT isolated words)
output_dir = "reference_audio_sentences"
os.makedirs(output_dir, exist_ok=True)

for idx, sentence in enumerate(sentences):
    audio_file = os.path.join(output_dir, f"sentence_{idx}.wav")
    generate_audio_with_piper(sentence, audio_file)

print(f"\nGenerated {len(sentences)} sentence audio files")


Audio generated: reference_audio_sentences\sentence_0.wav
Audio generated: reference_audio_sentences\sentence_1.wav
Audio generated: reference_audio_sentences\sentence_2.wav
Audio generated: reference_audio_sentences\sentence_3.wav
Audio generated: reference_audio_sentences\sentence_4.wav
Audio generated: reference_audio_sentences\sentence_5.wav
Audio generated: reference_audio_sentences\sentence_6.wav
Audio generated: reference_audio_sentences\sentence_7.wav
Audio generated: reference_audio_sentences\sentence_8.wav
Audio generated: reference_audio_sentences\sentence_9.wav
Audio generated: reference_audio_sentences\sentence_10.wav
Audio generated: reference_audio_sentences\sentence_11.wav
Audio generated: reference_audio_sentences\sentence_12.wav
Audio generated: reference_audio_sentences\sentence_13.wav
Audio generated: reference_audio_sentences\sentence_14.wav
Audio generated: reference_audio_sentences\sentence_15.wav
Audio generated: reference_audio_sentences\sentence_16.wav
Audio g

In [5]:
import torch
import librosa
import numpy as np
import whisper
from transformers import Wav2Vec2Processor, Wav2Vec2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)
wav2vec_model.eval()

whisper_model = whisper.load_model("large-v3")
print("Models loaded")


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models loaded


In [6]:
records = []

for idx, sentence in enumerate(sentences):
    print(f"\nProcessing Sentence {idx}: {sentence}")

    audio_file = os.path.join(output_dir, f"sentence_{idx}.wav")
    waveform, sr = librosa.load(audio_file, sr=16000)

    # Whisper transcription with word timestamps
    result = whisper_model.transcribe(audio_file, word_timestamps=True)
    words = [w for seg in result["segments"] for w in seg.get("words", [])]

    for word_index, w in enumerate(words):
        spoken_word = w["word"].strip(".,!?;: ").lower()

        start = int(w["start"] * sr)
        end = min(int(w["end"] * sr), len(waveform))
        chunk = waveform[start:end]

        if len(chunk) < 3200:
            chunk = np.pad(chunk, (0, 3200 - len(chunk)), mode="constant")

        inputs = processor(chunk, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            embedding = wav2vec_model(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

        phonemes = word_phonemes.get(spoken_word, "unknown")

        records.append({
            "id": f"{idx}_{word_index}",
            "values": embedding.tolist(),
            "metadata": {
                "sentence_id": idx,
                "sentence_text": sentence.lower(),
                "word_index": word_index,
                "word": spoken_word,
                "phonemes": phonemes
            }
        })
        print(f"  {spoken_word} -> embedded (id: {idx}_{word_index})")

print(f"\nTotal records: {len(records)}")



Processing Sentence 0: Meticulous researchers analyze unpredictable linguistic anomalies


d:\INFOSYS\.venv\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  meticulous -> embedded (id: 0_0)
  researchers -> embedded (id: 0_1)
  analyze -> embedded (id: 0_2)
  unpredictable -> embedded (id: 0_3)
  linguistic -> embedded (id: 0_4)
  anomalies -> embedded (id: 0_5)

Processing Sentence 1: Philosophical thinkers contemplate existential paradoxes silently


d:\INFOSYS\.venv\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  philosophical -> embedded (id: 1_0)
  thinkers -> embedded (id: 1_1)
  contemplate -> embedded (id: 1_2)
  existential -> embedded (id: 1_3)
  paradoxes -> embedded (id: 1_4)
  silently -> embedded (id: 1_5)

Processing Sentence 2: Honest knights write subtle rhythmic hymns
  honest -> embedded (id: 2_0)
  nights -> embedded (id: 2_1)
  writes -> embedded (id: 2_2)
  subtle -> embedded (id: 2_3)
  rhythmic -> embedded (id: 2_4)
  hymns -> embedded (id: 2_5)

Processing Sentence 3: Reading books improves knowledge and vocabulary skills
  reading -> embedded (id: 3_0)
  books -> embedded (id: 3_1)
  improves -> embedded (id: 3_2)
  knowledge -> embedded (id: 3_3)
  and -> embedded (id: 3_4)
  vocabulary -> embedded (id: 3_5)
  skills -> embedded (id: 3_6)

Processing Sentence 4: She speaks English clearly and very confidently
  she -> embedded (id: 4_0)
  speaks -> embedded (id: 4_1)
  english -> embedded (id: 4_2)
  clearly -> embedded (id: 4_3)
  and -> embedded (id: 4_4)
  very -> e

In [7]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "speech-therapy"

pc = Pinecone(api_key=API_KEY)
dense_index = pc.Index(INDEX_NAME)

# Upsert in batches
batch_size = 100
for i in range(0, len(records), batch_size):
    batch = records[i:i+batch_size]
    dense_index.upsert(vectors=batch, namespace="speech")

print("All records stored in namespace speech")


All records stored in namespace speech
